In [1]:
%pip install -q transformers


Note: you may need to restart the kernel to use updated packages.


In [5]:
import copy, gc, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, roc_auc_score

hf_logging.set_verbosity_error()          # hides the long LOAD REPORT / length warnings

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

df = pd.read_csv("data/daicwoz.csv")
train = df[df["split"] == "train"].reset_index(drop=True)
dev = df[df["split"] == "dev"].reset_index(drop=True)
test = df[df["split"] == "test"].reset_index(drop=True)
print(len(train), len(dev), len(test))

def evaluate(y_true, y_prob, thr=0.5):
    y_pred = (np.asarray(y_prob) >= thr).astype(int)
    pc, rc, fc, _ = precision_recall_fscore_support(y_true, y_pred, average=None, labels=[0, 1], zero_division=0)
    wp, wr, wf, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    return {"accuracy": accuracy_score(y_true, y_pred),
            "nd_precision": pc[0], "nd_recall": rc[0], "nd_f1": fc[0],      # non-depression class
            "precision": pc[1], "recall": rc[1], "f1": fc[1],               # depression class
            "macro_f1": f1_score(y_true, y_pred, average="macro"),
            "weighted_precision": wp, "weighted_recall": wr, "weighted_f1": wf,
            "auroc": roc_auc_score(y_true, y_prob)}

def best_threshold(y_true, y_prob):
    ts = np.linspace(0.05, 0.95, 91)
    scores = [f1_score(y_true, (np.asarray(y_prob) >= t).astype(int), average="macro") for t in ts]
    return ts[int(np.argmax(scores))]

device: cuda
107 35 47


In [6]:
def make_chunks(ids, size=510, stride=255):
    """Overlapping windows (50% overlap). 510 tokens + 2 special tokens = 512."""
    if len(ids) <= size:
        return [ids]
    chunks, start = [], 0
    while True:
        chunks.append(ids[start:start + size])
        if start + size >= len(ids):
            break
        start += stride
    return chunks

def build_samples(texts, labels, tokenizer, mode):
    """mode='base': first 510 tokens only. mode='window': all overlapping chunks."""
    cls_id = tokenizer.cls_token_id if tokenizer.cls_token_id is not None else tokenizer.bos_token_id
    sep_id = tokenizer.sep_token_id if tokenizer.sep_token_id is not None else tokenizer.eos_token_id
    samples = []                                    # (input_ids, label, doc_id)
    for doc_id, (text, label) in enumerate(zip(texts, labels)):
        ids = tokenizer(text, add_special_tokens=False)["input_ids"]
        chunks = [ids[:510]] if mode == "base" else make_chunks(ids)
        for ch in chunks:
            samples.append(([cls_id] + ch + [sep_id], int(label), doc_id))
    return samples

def make_collate(tokenizer):
    pad_id = tokenizer.pad_token_id
    def collate(batch):
        max_len = max(len(b[0]) for b in batch)
        input_ids = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
        attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)
        for i, b in enumerate(batch):
            input_ids[i, :len(b[0])] = torch.tensor(b[0])
            attention_mask[i, :len(b[0])] = 1
        return {"input_ids": input_ids, "attention_mask": attention_mask,
                "labels": torch.tensor([b[1] for b in batch])}
    return collate

def predict(model, samples, n_docs, collate, batch_size=8):
    """Probability of depression per document (window: mean over its chunks)."""
    model.eval()
    probs = []
    with torch.no_grad():
        for batch in DataLoader(samples, batch_size=batch_size, shuffle=False, collate_fn=collate):
            inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            probs += torch.softmax(model(**inputs).logits, dim=-1)[:, 1].cpu().tolist()
    doc_ids = np.array([s[2] for s in samples])
    return np.bincount(doc_ids, weights=probs, minlength=n_docs) / np.bincount(doc_ids, minlength=n_docs)

def train_one(model_name, mode, lr, epochs=10, patience=3, batch_size=4, seed=42):
    set_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    collate = make_collate(tokenizer)
    tr = build_samples(train["text"], train["label"], tokenizer, mode)
    dv = build_samples(dev["text"], dev["label"], tokenizer, mode)
    te = build_samples(test["text"], test["label"], tokenizer, mode)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
    counts = np.bincount(train["label"])
    class_w = torch.tensor(len(train) / (2 * counts), dtype=torch.float).to(device)   # class imbalance
    loss_fn = nn.CrossEntropyLoss(weight=class_w)

    loader = DataLoader(tr, batch_size=batch_size, shuffle=True, collate_fn=collate)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total = len(loader) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)

    best_auc, best_state, bad = -1, None, 0
    for epoch in range(epochs):
        model.train()
        for batch in loader:
            labels = batch["labels"].to(device)
            inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            loss = loss_fn(model(**inputs).logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
        auc = roc_auc_score(dev["label"], predict(model, dv, len(dev), collate))
        if auc > best_auc:
            best_auc, bad = auc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)                      # best epoch on dev
    dev_prob = predict(model, dv, len(dev), collate)
    test_prob = predict(model, te, len(test), collate)
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return best_auc, dev_prob, test_prob

In [7]:
MODELS = {"BERT": "bert-base-uncased", "RoBERTa": "roberta-base"}
MODES = ["base", "window"]
LRS = [1e-5, 2e-5, 5e-5]

rows = []
for name, hf_name in MODELS.items():
    for mode in MODES:
        best = None
        for lr in LRS:
            t0 = time.time()
            auc, dev_prob, test_prob = train_one(hf_name, mode, lr)
            print(f"{name}-{mode} lr={lr}: dev AUROC={auc:.3f}  ({time.time() - t0:.0f}s)")
            if best is None or auc > best[0]:
                best = (auc, lr, dev_prob, test_prob)
        auc, lr, dev_prob, test_prob = best
        thr = best_threshold(dev["label"], dev_prob)       # threshold chosen on dev
        row = {"model": f"{name}-{mode}", "lr": lr, "thr": round(thr, 2), "dev_auroc": round(auc, 3)}
        row.update({k: round(v, 3) for k, v in evaluate(test["label"], test_prob, thr).items()})
        rows.append(row)

results = pd.DataFrame(rows)
results.to_csv("daicwoz_transformer_results.csv", index=False)
results

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BERT-base lr=1e-05: dev AUROC=0.533  (13s)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BERT-base lr=2e-05: dev AUROC=0.370  (13s)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BERT-base lr=5e-05: dev AUROC=0.529  (11s)


... (15 outputs left out)

RoBERTa-window lr=2e-05: dev AUROC=0.670  (53s)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RoBERTa-window lr=5e-05: dev AUROC=0.688  (78s)


,model,lr,thr,dev_auroc,accuracy,nd_precision,nd_recall,nd_f1,precision,recall,f1,macro_f1,weighted_precision,weighted_recall,weighted_f1,auroc
0,BERT-base,0.00001,0.48,0.533,0.660,0.707,0.879,0.784,0.333,0.143,0.200,0.492,0.596,0.660,0.610,0.587
1,BERT-window,0.00005,0.30,0.786,0.617,0.714,0.758,0.735,0.333,0.286,0.308,0.521,0.601,0.617,0.608,0.665
2,RoBERTa-base,0.00002,0.39,0.667,0.787,0.795,0.939,0.861,0.750,0.429,0.545,0.703,0.782,0.787,0.767,0.736
3,RoBERTa-window,0.00001,0.21,0.830,0.681,0.821,0.697,0.754,0.474,0.643,0.545,0.650,0.718,0.681,0.692,0.749


In [8]:
SEEDS = [0, 1, 2, 3, 4]
best_lr = {r["model"]: r["lr"] for _, r in results.iterrows()}   # best lr from Cell 4

runs = []
for name, hf_name in MODELS.items():
    for mode in MODES:
        key = f"{name}-{mode}"
        for seed in SEEDS:
            auc, dev_prob, test_prob = train_one(hf_name, mode, best_lr[key], seed=seed)
            thr = best_threshold(dev["label"], dev_prob)
            m = evaluate(test["label"], test_prob, thr)
            runs.append({"model": key, "seed": seed, **m})
            print(f"{key} seed={seed}: test AUROC={m['auroc']:.3f}  macro_f1={m['macro_f1']:.3f}")

runs_df = pd.DataFrame(runs)
runs_df.to_csv("daicwoz_multiseed_runs.csv", index=False)

metrics = ["accuracy", "nd_precision", "nd_recall", "nd_f1", "precision", "recall", "f1",
           "macro_f1", "weighted_precision", "weighted_recall", "weighted_f1", "auroc"]
g = runs_df.groupby("model", sort=False)[metrics]
summary = g.mean().round(3).astype(str) + " ± " + g.std().round(3).astype(str)
summary

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BERT-base seed=0: test AUROC=0.667  macro_f1=0.603


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BERT-base seed=1: test AUROC=0.760  macro_f1=0.710


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BERT-base seed=2: test AUROC=0.675  macro_f1=0.521


... (31 outputs left out)

RoBERTa-window seed=3: test AUROC=0.747  macro_f1=0.613


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RoBERTa-window seed=4: test AUROC=0.838  macro_f1=0.706


,accuracy,nd_precision,nd_recall,nd_f1,precision,recall,f1,macro_f1,weighted_precision,weighted_recall,weighted_f1,auroc
model,,,,,,,,,,,,
BERT-base,0.638 ± 0.096,0.754 ± 0.105,0.745 ± 0.149,0.74 ± 0.073,0.484 ± 0.32,0.386 ± 0.326,0.343 ± 0.223,0.542 ± 0.121,0.674 ± 0.134,0.638 ± 0.096,0.622 ± 0.089,0.658 ± 0.109
BERT-window,0.643 ± 0.059,0.819 ± 0.088,0.667 ± 0.215,0.71 ± 0.093,0.345 ± 0.198,0.586 ± 0.355,0.43 ± 0.245,0.57 ± 0.097,0.678 ± 0.113,0.643 ± 0.059,0.627 ± 0.056,0.716 ± 0.064
RoBERTa-base,0.677 ± 0.023,0.738 ± 0.063,0.867 ± 0.163,0.785 ± 0.044,0.177 ± 0.242,0.229 ± 0.329,0.197 ± 0.273,0.491 ± 0.116,0.571 ± 0.113,0.677 ± 0.023,0.61 ± 0.053,0.719 ± 0.044
RoBERTa-window,0.685 ± 0.065,0.812 ± 0.039,0.727 ± 0.153,0.758 ± 0.078,0.501 ± 0.077,0.586 ± 0.185,0.517 ± 0.082,0.637 ± 0.051,0.719 ± 0.025,0.685 ± 0.065,0.686 ± 0.056,0.748 ± 0.062
